In [1]:
# INSTALL DEPENDENCIES
!apt-get update -qq
!apt-get install -y wkhtmltopdf
!pip install -q yfinance plotly pdfkit jinja2 kaleido==0.2.1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wkhtmltopdf is already the newest version (0.12.6-2).
0 upgraded, 0 newly installed, 0 to remove and 44 not upgraded.


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import pdfkit
from jinja2 import Template
import base64
import warnings

# Suppress pandas FutureWarnings from yfinance
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configuration for pdfkit to find the installed binary in Colab
path_wkhtmltopdf = '/usr/bin/wkhtmltopdf'
config = pdfkit.configuration(wkhtmltopdf=path_wkhtmltopdf)

print("Dalal Street Valuation Engine configured successfully.")

Dalal Street Valuation Engine configured successfully.


In [3]:
class IndianFinancialDataEngine:
    """
    Handles extraction and standardization of fundamental data from yfinance for NSE stocks.
    Converts metrics to INR Crores where applicable.
    """
    def __init__(self, ticker_symbol: str):
        # Ensure it has the Indian exchange suffix if not provided
        if not ticker_symbol.endswith('.NS') and not ticker_symbol.endswith('.BO'):
            ticker_symbol += '.NS'

        self.ticker_symbol = ticker_symbol
        self.ticker = yf.Ticker(ticker_symbol)
        self.info = self.ticker.info

    def fetch_historical_financials(self) -> pd.DataFrame:
        """Fetches and merges Income Statement, Balance Sheet, and Cash Flow."""
        try:
            inc = self.ticker.financials.T
            bs = self.ticker.balance_sheet.T
            cf = self.ticker.cashflow.T

            df = inc.join(bs, how='outer').join(cf, how='outer', rsuffix='_cf')
            df.sort_index(ascending=True, inplace=True)
            return df
        except Exception as e:
            raise ValueError(f"Failed to fetch data for {self.ticker_symbol}: {e}")

    def extract_core_metrics(self, df: pd.DataFrame) -> dict:
        """Extracts the latest metrics needed for a DCF, localizing to Indian standards."""
        latest = df.iloc[-1]

        current_price = self.info.get('currentPrice', self.info.get('regularMarketPrice', 0))
        shares_out = self.info.get('sharesOutstanding', 0)

        # Indian standard corporate tax regime is approx 25.17% for domestic companies
        tax_rate = self.info.get('taxRate', 0.2517)

        # Absolute figures
        market_cap_abs = current_price * shares_out
        total_debt_abs = self.info.get('totalDebt', latest.get('Total Debt', 0))
        cash_abs = self.info.get('totalCash', latest.get('Cash And Cash Equivalents', 0))
        revenue_abs = latest.get('Total Revenue', 0)
        ebit_abs = latest.get('EBIT', 0)

        # Convert to Crores (1 Crore = 10,000,000) for reporting
        to_crores = 10_000_000

        return {
            "current_price": current_price,
            "shares_out": shares_out,
            "market_cap_abs": market_cap_abs,
            "total_debt_abs": total_debt_abs,
            "cash_abs": cash_abs,
            "revenue_abs": revenue_abs,
            "ebit_abs": ebit_abs,

            # Formatted for institutional reporting
            "market_cap_cr": market_cap_abs / to_crores,
            "total_debt_cr": total_debt_abs / to_crores,
            "cash_cr": cash_abs / to_crores,
            "revenue_cr": revenue_abs / to_crores,

            "tax_rate": tax_rate,
            "beta": self.info.get('beta', 1.0)
        }

In [4]:
class IndianDCFModeler:
    """
    Calculates WACC and computes intrinsic value using Indian Macroeconomic assumptions.
    """
    def __init__(self, metrics: dict,
                 india_10y_gsec: float = 0.0684, # Approx 6.84% current Indian 10Y Yield
                 india_erp: float = 0.0700):     # Standard Equity Risk Premium for India
        self.metrics = metrics
        self.rf = india_10y_gsec
        self.erp = india_erp

    def calculate_wacc(self, cost_of_debt: float = 0.085) -> float:
        """Calculates Weighted Average Cost of Capital. Assumes higher Indian borrowing cost."""
        E = self.metrics['market_cap_abs']
        D = self.metrics['total_debt_abs']
        V = E + D

        if V == 0:
            return 0.12 # Fallback WACC for Indian equities

        # Cost of Equity (CAPM)
        Re = self.rf + self.metrics['beta'] * self.erp

        # Cost of Debt after tax
        Rd = cost_of_debt * (1 - self.metrics['tax_rate'])

        wacc = (E/V * Re) + (D/V * Rd)
        return wacc

    def project_fcff(self, years: int = 5, rev_growth: float = 0.10, ebit_margin: float = 0.18):
        """Projects Free Cash Flow to Firm under simplified assumptions."""
        projections = []
        current_rev = self.metrics['revenue_abs']

        for year in range(1, years + 1):
            rev = current_rev * ((1 + rev_growth) ** year)
            ebit = rev * ebit_margin
            nopat = ebit * (1 - self.metrics['tax_rate'])

            # Proxy: Assuming D&A offsets CapEx & NWC changes for maturity
            fcff = nopat
            projections.append(fcff)

        return projections

    def compute_intrinsic_value(self, fcff_projections: list, wacc: float, terminal_growth: float = 0.045) -> float:
        """Calculates Present Value to find Share Price. Terminal growth adjusted for India (~4.5%)."""

        # --- THE SAFETY FIX ---
        # If growth is greater than or equal to WACC, the math breaks. Return NaN.
        if terminal_growth >= wacc:
            return np.nan
        # ----------------------

        pv_fcff = 0
        for i, cf in enumerate(fcff_projections):
            pv_fcff += cf / ((1 + wacc) ** (i + 1))

        final_year_cf = fcff_projections[-1]
        terminal_value = (final_year_cf * (1 + terminal_growth)) / (wacc - terminal_growth)
        pv_tv = terminal_value / ((1 + wacc) ** len(fcff_projections))

        enterprise_value = pv_fcff + pv_tv
        equity_value = enterprise_value - self.metrics['total_debt_abs'] + self.metrics['cash_abs']

        if self.metrics['shares_out'] == 0: return 0
        return equity_value / self.metrics['shares_out']

In [5]:
class VisualizationEngine:
    """Generates professional quantitative charts optimized for INR display."""

    @staticmethod
    def sensitivity_heatmap(model: IndianDCFModeler, wacc_base: float, g_base: float) -> str:
        wacc_range = np.linspace(wacc_base - 0.02, wacc_base + 0.02, 5)
        g_range = np.linspace(g_base - 0.015, g_base + 0.015, 5) # Wider range for India

        prices = np.zeros((len(wacc_range), len(g_range)))

        for i, w in enumerate(wacc_range):
            for j, g in enumerate(g_range):
                projections = model.project_fcff()
                prices[i, j] = model.compute_intrinsic_value(projections, w, g)

        fig = go.Figure(data=go.Heatmap(
            z=prices,
            x=[f"{g*100:.1f}%" for g in g_range],
            y=[f"{w*100:.1f}%" for w in wacc_range],
            colorscale='RdYlGn',
            text=np.round(prices, 1),
            texttemplate="₹%{text}"
        ))

        fig.update_layout(
            title="Implied Share Price Sensitivity (₹)",
            xaxis_title="Terminal Growth Rate (g)",
            yaxis_title="WACC",
            width=600, height=400,
            plot_bgcolor='white'
        )

        img_bytes = fig.to_image(format="png")
        return base64.b64encode(img_bytes).decode("utf-8")

    @staticmethod
    def football_field(ticker: str, current: float, dcf_bear: float, dcf_base: float, dcf_bull: float) -> str:
        fig = go.Figure()

        fig.add_trace(go.Bar(
            y=['DCF Valuation'],
            x=[dcf_bull - dcf_bear],
            base=[dcf_bear],
            orientation='h',
            marker_color='rgba(255, 153, 51, 0.7)', # Indian Saffron inspired color
            name='DCF Range'
        ))

        fig.add_vline(x=current, line_width=3, line_dash="dash", line_color="#138808", # India Green
                      annotation_text=f"CMP: ₹{current:.1f}")

        fig.update_layout(
            title=f"{ticker} Valuation Summary",
            barmode='overlay',
            showlegend=False,
            width=600, height=300,
            xaxis_title="Share Price (₹)",
            plot_bgcolor='white'
        )

        img_bytes = fig.to_image(format="png")
        return base64.b64encode(img_bytes).decode("utf-8")

In [6]:
html_template = """
<!DOCTYPE html>
<html>
<head>
    <style>
        body { font-family: 'Helvetica', sans-serif; color: #333; margin: 40px; }
        h1 { color: #000080; border-bottom: 2px solid #FF9933; padding-bottom: 10px; }
        .metric-box { background: #f7fafc; padding: 15px; border-radius: 5px; margin-bottom: 20px; border-left: 5px solid #138808; }
        .grid { display: table; width: 100%; }
        .col { display: table-cell; width: 50%; padding: 10px; }
        img { max-width: 100%; height: auto; border: 1px solid #e2e8f0; }
    </style>
</head>
<body>
    <h1>Institutional Equity Research: {{ ticker }}</h1>

    <div class="metric-box">
        <h3>Current Market Overview (NSE)</h3>
        <p><strong>Current Market Price (CMP):</strong> ₹{{ "%.2f"|format(metrics.current_price) }}</p>
        <p><strong>Market Cap:</strong> ₹{{ "{:,.1f}".format(metrics.market_cap_cr) }} Cr.</p>
        <p><strong>Total Debt:</strong> ₹{{ "{:,.1f}".format(metrics.total_debt_cr) }} Cr.</p>
        <p><strong>Implied Base Value:</strong> ₹{{ "%.2f"|format(base_value) }}</p>
        <p><strong>Upside/Downside:</strong> {{ "%.1f"|format(((base_value / metrics.current_price) - 1) * 100) }}%</p>
    </div>

    <div class="grid">
        <div class="col">
            <h3>Valuation Football Field</h3>
            <img src="data:image/png;base64,{{ football_img }}" />
        </div>
        <div class="col">
            <h3>DCF Sensitivity Analysis</h3>
            <img src="data:image/png;base64,{{ heatmap_img }}" />
        </div>
    </div>

    <h3>Methodology & Macro Assumptions</h3>
    <p>The Discounted Cash Flow (DCF) analysis incorporates Indian macroeconomic fundamentals, assuming a Risk-Free Rate of 6.84% (10Y G-Sec), an Equity Risk Premium of 7.00%, and a statutory corporate tax rate of 25.17%. The calculated WACC is {{ "%.1f"|format(wacc * 100) }}% with a terminal growth rate of 4.5% to reflect long-term GDP growth projections.</p>
</body>
</html>
"""

def generate_pdf_report(data_dict: dict, output_filename: str):
    template = Template(html_template)
    html_out = template.render(**data_dict)

    options = {
        'page-size': 'A4',
        'margin-top': '0.75in',
        'margin-right': '0.75in',
        'margin-bottom': '0.75in',
        'margin-left': '0.75in',
        'encoding': "UTF-8",
        'enable-local-file-access': None
    }

    pdfkit.from_string(html_out, output_filename, options=options, configuration=config)
    print(f"Report successfully saved to {output_filename}")

In [10]:
def run_pipeline(ticker_symbol: str):
    print(f"Initiating Dalal Street pipeline for {ticker_symbol}...")

    # 1. Fetch Data
    engine = IndianFinancialDataEngine(ticker_symbol)
    df = engine.fetch_historical_financials()
    metrics = engine.extract_core_metrics(df)
    print("Data extraction and INR Crores conversion complete.")

    # 2. Build Model
    model = IndianDCFModeler(metrics)
    wacc = model.calculate_wacc()

    # Scenarios (Adjusted for typical Indian large-cap growth)
    cf_base = model.project_fcff(rev_growth=0.10, ebit_margin=0.18)
    cf_bear = model.project_fcff(rev_growth=0.06, ebit_margin=0.14)
    cf_bull = model.project_fcff(rev_growth=0.14, ebit_margin=0.22)

    base_value = model.compute_intrinsic_value(cf_base, wacc, terminal_growth=0.045)
    bear_value = model.compute_intrinsic_value(cf_bear, wacc + 0.015, terminal_growth=0.035)
    bull_value = model.compute_intrinsic_value(cf_bull, wacc - 0.015, terminal_growth=0.055)
    print(f"DCF computed. Base Implied Value: ₹{base_value:.2f}")

    # 3. Visualizations
    heatmap_img = VisualizationEngine.sensitivity_heatmap(model, wacc, 0.045)
    football_img = VisualizationEngine.football_field(
        engine.ticker_symbol, metrics['current_price'], bear_value, base_value, bull_value
    )
    print("Visualizations generated.")

    # 4. Generate PDF
    report_data = {
        "ticker": engine.ticker_symbol,
        "metrics": metrics,
        "base_value": base_value,
        "wacc": wacc,
        "heatmap_img": heatmap_img,
        "football_img": football_img
    }

    # Clean ticker name for filename (removes .NS)
    clean_ticker = engine.ticker_symbol.replace('.NS', '').replace('.BO', '')
    output_file = f"{clean_ticker}_Initiation_Report.pdf"

    generate_pdf_report(report_data, output_file)

# RUN THE PIPELINE (Test with Indian Large Caps)
try:
    # Example: Reliance Industries, TCS, Infosys, or ITC
    run_pipeline("MARUTI.NS")
except Exception as e:
    print(f"Pipeline failed: {e}")

Initiating Dalal Street pipeline for MARUTI.NS...
Data extraction and INR Crores conversion complete.
DCF computed. Base Implied Value: ₹24700.07
Visualizations generated.
Report successfully saved to MARUTI_Initiation_Report.pdf
